# 제3장: AI 윤리 실무화

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch03.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

### AI 윤리 원칙의 운용화(Operationalization)

## Abstract적 원칙에서 측정 가능한 지표로

### 공정성(공정성)의 지표화

**Korean Context Fairness Evaluator**

In [ ]:
import pandas as pd
from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate
# Requirement: fairlearn==0.10.1

def evaluate_korean_fairness(y_true, y_pred, sensitive_features):
 """
 Evaluate fairness considering Korean specific proxies.
 Sensitive Features:
 - Standard: Age, Gender
 - Proxies: Military Service (Gender), Univ. Entrance Year (Age)
 """
 metrics = {
 "selection_rate": selection_rate,
 "false_positive_rate": false_positive_rate
 }
 
 # Analyze disparities across intersectional groups
 mf = MetricFrame(
 metrics=metrics,
 y_true=y_true,
 y_pred=y_pred,
 sensitive_features=sensitive_features
 )
 
 # Check for 'Disparate Impact' (KR Labor Standards Act context)
 print("Fairness Audit Report (2026 Standard):")
 print(mf.by_group)
 return mf

### 설명가능성(설명가능성)의 지표화

### 책임성(책임성)의 조직적 구현

**Human Oversight Manager**

In [ ]:
from datetime import datetime, timedelta

class HumanDecisionRecord:
    """
    인간 의사결정 기록 클래스. AI 추천과 인간 최종 결정을 함께 저장하여
    책임성 추적과 법적 보존 요건(5년)을 충족한다.
    """
    def __init__(self, record_id, timestamp, model_id, ai_recommendation,
                 human_decision, human_rationale, reviewer_id):
        """
        기록 초기화. Args: record_id(기록ID), timestamp(datetime), model_id,
        ai_recommendation(AI 추천값), human_decision(인간 최종 결정),
        human_rationale(결정 근거 - 필수), reviewer_id(검토자 ID).
        """
        self.record_id = record_id
        self.timestamp = timestamp
        self.model_id = model_id
        self.ai_recommendation = ai_recommendation
        self.human_decision = human_decision
        self.human_rationale = human_rationale  # 결정 근거 (법적 필수 항목)
        self.reviewer_id = reviewer_id
        # 보존 메타데이터: 개인정보보호법상 5년 보존 의무
        self.retention_until = timestamp + timedelta(days=365 * 5)

class HumanOversightManager:
    """
    인간 감독 관리자 클래스. AI 자동화 편향(Automation Bias)을 탐지하고
    인간 검토자가 AI 추천을 무비판적으로 수용하는지 모니터링한다.
    """

    def analyze_automation_bias(self, records, threshold=0.95):
        """
        자동화 편향 탐지. AI 추천과 인간 결정의 일치율이 threshold를 초과하면
        경고를 발생시킨다. Args: records(HumanDecisionRecord 목록),
        threshold(경고 임계값, 기본 0.95). Returns: None.
        """
        agreement_count = sum(1 for r in records if r.agreed_with_ai)
        rate = agreement_count / len(records)
        if rate > threshold:
            print(f"경고: 일치율({rate:.1%})이 임계값을 초과합니다. 자동화 편향 의심.")

### 안전성(안전성)과 Security(보안)의 기술적 요구사항

**Safety Evaluator**

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score
from fairlearn.metrics import demographic_parity_difference  # 인구통계 동등성 차이 측정
def evaluate_noise_robustness(model, X, y, noise_level=0.1):
 """Evaluate robustness against noise"""
 clean_acc = accuracy_score(y, model.predict(X))
 noise = np.random.normal(0, noise_level, X.shape)
 noisy_acc = accuracy_score(y, model.predict(X + noise))
 
 degradation = (clean_acc - noisy_acc) / clean_acc
 print(f"Performance degradation: {degradation:.1%}")
 return degradation < 0.05 # Recommendation: < 5% degradation

### Privacy(Privacy) 보호 메커니즘

**Differential Privacy Application Example**

In [ ]:
from diffprivlib.models import LogisticRegression

def train_private_model(X, y, epsilon=1.0):
 """Train Diff-Privacy Logistic Regression"""
 # Smaller epsilon = Higher privacy, Lower accuracy
 dp_model = LogisticRegression(epsilon=epsilon, data_norm=10)
 dp_model.fit(X, y)
 return dp_model

## Principle 간 충돌과 트레이드오프 관리

### 공정성 vs 정확도

**Fairness Constrained Learning with Fairlearn**

In [ ]:
from fairlearn.reductions import (
 ExponentiatedGradient, 
 DemographicParity,
 EqualizedOdds,
)
from sklearn.linear_model import LogisticRegression

# Base Estimator
base_estimator = LogisticRegression(solver='lbfgs', max_iter=1000)

# Define Fairness Constraint
constraint = DemographicParity(difference_bound=0.05)
# Or: constraint = EqualizedOdds(difference_bound=0.05)

# Train with Fairness Constraint
mitigator = ExponentiatedGradient(
 estimator=base_estimator,
 constraints=constraint,
 eps=0.01, # Constraint violation tolerance
)

mitigator.fit(X_train, y_train, sensitive_features=sensitive_train)

# Prediction
y_pred_fair = mitigator.predict(X_test)

**Threshold Optimization by Group**

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score
from fairlearn.metrics import demographic_parity_difference  # 인구통계 동등성 차이 측정
from fairlearn.postprocessing import ThresholdOptimizer

# Create ThresholdOptimizer
postprocess = ThresholdOptimizer(
 estimator=trained_model,
 constraints="demographic_parity",
 objective="accuracy_score",
 prefit=True,
)

# Fit ThresholdOptimizer
postprocess.fit(X_train, y_train, sensitive_features=sensitive_train)

# Predict with fairness
y_pred_fair = postprocess.predict(X_test, sensitive_features=sensitive_test)

**Pareto Frontier Exploration**

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score
from fairlearn.metrics import demographic_parity_difference  # 인구통계 동등성 차이 측정
def explore_pareto_frontier(
    model_class, X_train, y_train, sensitive_train,
    X_test, y_test, sensitive_test,
    epsilon_values=[0.01, 0.02, 0.05, 0.1, 0.2]
):
    """Explore fairness-accuracy tradeoff"""
    results = []
    for eps in epsilon_values:
        # Train with Fairness Constraint
        const = DemographicParity(difference_bound=eps)
        mitigator = ExponentiatedGradient(
            estimator=model_class(),
            constraints=const, eps=eps
        )
        mitigator.fit(X_train, y_train, 
                      sensitive_features=sensitive_train)
        
        # Prediction and Metrics
        y_pred = mitigator.predict(X_test)
        auc = roc_auc_score(y_test, mitigator.predict(X_test))
        dp_diff = demographic_parity_difference(
            y_test, y_pred, sensitive_features=sensitive_test
        )
        results.append({'eps': eps, 'auc': auc, 'dp': dp_diff})
    return results

### 투명성 vs 영업비밀/Privacy

**SHAP with Differential Privacy**

In [ ]:
def dp_shap_explanation(explainer, instance, epsilon=1.0, sensitivity=0.1):
    """SHAP values with Differential Privacy noise"""
    shap_values = explainer.shap_values(instance.reshape(1, -1))
    if isinstance(shap_values, list):
        shap_values = shap_values[1] # Binary Class 1
    
    shap_values = shap_values.flatten()
    # Add Laplace noise for privacy
    scale = sensitivity / epsilon
    noise = np.random.laplace(0, scale, shap_values.shape)
    return shap_values + noise

**Aggregated Feature Importance**

In [ ]:
def generate_aggregated_explanation(shap_values, feature_names, min_size=100):
    """Aggregate explanation for K-Anonymity"""
    if len(shap_values) < min_size:
        raise ValueError(f"Sample size < {min_size}")
    
    explanation = {}
    for i, feat in enumerate(feature_names):
        feat_shap = shap_values[:, i]
        explanation[feat] = {
            'abs_impact': float(np.abs(feat_shap).mean()),
            'p50': float(np.percentile(feat_shap, 50))
        }
    return explanation

**Generating Surrogate Model for Explanation**

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score
from fairlearn.metrics import demographic_parity_difference  # 인구통계 동등성 차이 측정
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

def create_explainable_surrogate(
 original_model,
 X_train: np.ndarray,
 max_depth: int = 5,
 min_fidelity: float = 0.9,
) -> tuple:
 """
 Create explainable surrogate for original model
 
 Parameters
 ----------
 original_model : Any
 Original black-box model
 X_train : np.ndarray
 Learning Data
 max_depth : int
 ( )
 min_fidelity : float
 Model Minimum days 
 
 Returns
 -------
 tuple
 ( Model, )
 """
 # Use original model predictions as labels
 y_original = original_model.predict(X_train)
 
 # Train interpretable surrogate model
 surrogate = DecisionTreeClassifier(
 max_depth=max_depth,
 min_samples_leaf=50, # Prevent overfitting
 random_state=42,
 )
 surrogate.fit(X_train, y_original)
 
 # Calculate fidelity
 y_surrogate = surrogate.predict(X_train)
 fidelity = accuracy_score(y_original, y_surrogate)
 
 if fidelity < min_fidelity:
 raise ValueError(
 f" Model ({fidelity:.2%}) "
 f"Minimum ({min_fidelity:.0%}) . "
 f"max_depth Model ."
 )
 
 return surrogate, fidelity

### 개인화 vs 차별 금지

**Proxy Discrimination Detection and Feature Removal**

In [ ]:
import numpy as np
import pandas as pd
from typing import List, Tuple

class ProxyDiscriminationDetector:
 """
 Detect and mitigate proxy discrimination
 
 Identify features correlated with sensitive attributes
 and exclude or adjust them from personalization systems
 """
 
 def __init__(
 self,
 sensitive_attrs: List[str],
 correlation_threshold: float = 0.3,
 ):
 """
 Parameters
 ----------
 sensitive_attrs : list
 List of sensitive attributes
 correlation_threshold : float
 Correlation threshold for proxy detection
 """
 self.sensitive_attrs = sensitive_attrs
 self.correlation_threshold = correlation_threshold
 self.proxy_map = {}
 
 def fit(
 self,
 X: pd.DataFrame,
 sensitive_df: pd.DataFrame,
 ) -> 'ProxyDiscriminationDetector':
 """
 Learn proxy relationships
 
 Parameters
 ----------
 X : pd.DataFrame
 Feature data
 sensitive_df : pd.DataFrame
 Sensitive attribute data
 """
 self.proxy_map = {}
 
 # Encode sensitive attributes
 sensitive_numeric = sensitive_df.copy()
 for col in sensitive_numeric.columns:
 if sensitive_numeric[col].dtype == 'object':
 sensitive_numeric[col] = pd.factorize(sensitive_numeric[col])[0]
 
 # Encode features
 X_numeric = X.copy()
 for col in X_numeric.columns:
 if X_numeric[col].dtype == 'object':
 X_numeric[col] = pd.factorize(X_numeric[col])[0]
 
 # Analyze correlations
 for sens_attr in self.sensitive_attrs:
 if sens_attr not in sensitive_numeric.columns:
 continue
 
 proxies = []
 sens_values = sensitive_numeric[sens_attr]
 
 for feat in X_numeric.columns:
 try:
 corr = sens_values.corr(X_numeric[feat])
 if pd.notna(corr) and abs(corr) >= self.correlation_threshold:
 proxies.append({
 'feature': feat,
 'correlation': corr,
 'strength': 'strong' if abs(corr) >= 0.5 else 'moderate',
 })
 except Exception:
 continue
 
 if proxies:
 self.proxy_map[sens_attr] = sorted(
 proxies,
 key=lambda x: abs(x['correlation']),
 reverse=True
 )
 
 return self
 
 def get_safe_features(
 self,
 all_features: List[str],
 remove_strong_only: bool = False,
 ) -> Tuple[List[str], List[str]]:
 """
 Return list of safe features (non-proxies)
 
 Parameters
 ----------
 all_features : list
 List of all features
 remove_strong_only : bool
 Whether to remove only strong proxies
 
 Returns
 -------
 tuple
 ( , )
 """
 proxy_features = set()
 
 for sens_attr, proxies in self.proxy_map.items():
 for proxy in proxies:
 if remove_strong_only and proxy['strength'] != 'strong':
 continue
 proxy_features.add(proxy['feature'])
 
 safe_features = [f for f in all_features if f not in proxy_features]
 removed_features = list(proxy_features)
 
 return safe_features, removed_features
 
 def generate_report(self) -> str:
 """Generate proxy discrimination report"""
 report = "# Proxy Discrimination Analysis Report\n\n"
 
 if not self.proxy_map:
 report += "No proxy relationships detected.\n"
 return report
 
 for sens_attr, proxies in self.proxy_map.items():
 report += f"## Sensitive Attribute: {sens_attr}\n\n"
 report += "| Feature | Correlation | Strength | Recommendation |\n"
 report += "|---------|-------------|----------|----------------|\n"
 
 for proxy in proxies:
 action = "Remove" if proxy['strength'] == 'strong' else "Monitor"
 report += (
 f"| {proxy['feature']} | "
 f"{proxy['correlation']:.3f} | "
 f"{proxy['strength']} | "
 f"{action} |\n"
 )
 
 report += "\n"
 
 return report

### 의사결정 프레임워크: 원칙 우선순위 결정 방법론

**Principle Priority Scoring**

In [ ]:
from typing import Dict, List
from dataclasses import dataclass

@dataclass
class PrincipleWeight:
 """Weights for each principle"""
 fairness: float = 0.0
 transparency: float = 0.0
 accountability: float = 0.0
 safety: float = 0.0
 privacy: float = 0.0
 accuracy: float = 0.0
 
 def normalize(self) -> 'PrincipleWeight':
 """Normalize weights (sum to 1)"""
 total = (
 self.fairness + self.transparency + self.accountability +
 self.safety + self.privacy + self.accuracy
 )
 if total == 0:
 return self
 return PrincipleWeight(
 fairness=self.fairness / total,
 transparency=self.transparency / total,
 accountability=self.accountability / total,
 safety=self.safety / total,
 privacy=self.privacy / total,
 accuracy=self.accuracy / total,
 )


def calculate_principle_priority(
 context: Dict[str, any],
 stakeholder_weights: Dict[str, float],
 stakeholder_priorities: Dict[str, PrincipleWeight],
) -> PrincipleWeight:
 """
 Calculate priority based on context and stakeholders
 
 Parameters
 ----------
 context : dict
 Context analysis result {'high_risk': bool, 'impact_severity': float, ...}
 stakeholder_weights : dict
 Stakeholder weights {'user': 0.3, 'business': 0.3, 'regulator': 0.4}
 stakeholder_priorities : dict
 Stakeholder principle priorities
 
 Returns
 -------
 PrincipleWeight
 Combined principle priority
 """
 # Initialize base weights
 combined = PrincipleWeight()
 
 # Aggregate stakeholder weights
 for stakeholder, weight in stakeholder_weights.items():
 if stakeholder not in stakeholder_priorities:
 continue
 
 sp = stakeholder_priorities[stakeholder]
 combined.fairness += weight * sp.fairness
 combined.transparency += weight * sp.transparency
 combined.accountability += weight * sp.accountability
 combined.safety += weight * sp.safety
 combined.privacy += weight * sp.privacy
 combined.accuracy += weight * sp.accuracy
 
 # Adjust based on context
 if context.get('high_risk', False):
 # High-risk: Boost Fairness, Accountability, Safety
 combined.fairness *= 1.5
 combined.accountability *= 1.3
 combined.safety *= 1.3
 
 if context.get('privacy_sensitive', False):
 # Privacy sensitive: Boost Privacy
 combined.privacy *= 1.5
 
 if context.get('high_impact', False):
 # High impact: Boost Transparency, Accountability
 combined.transparency *= 1.3
 combined.accountability *= 1.3
 
 return combined.normalize()


# Example
def example_priority_calculation():
 """Example: Recruiting AI System priority calculation"""
 
 # Context Analysis
 context = {
 'high_risk': True, # High-risk area: Employment/Recruiting
 'privacy_sensitive': True, # Processing personal data
 'high_impact': True, # Impact on employment
 }
 
 # Stakeholder Weights
 stakeholder_weights = {
 'job_applicants': 0.35,
 'hr_team': 0.25,
 'regulator': 0.25,
 'company_leadership': 0.15,
 }
 
 # Stakeholder Priorities
 stakeholder_priorities = {
 'job_applicants': PrincipleWeight(
 fairness=0.35, transparency=0.25, privacy=0.20,
 accountability=0.10, safety=0.05, accuracy=0.05
 ),
 'hr_team': PrincipleWeight(
 accuracy=0.30, fairness=0.25, transparency=0.15,
 accountability=0.15, safety=0.10, privacy=0.05
 ),
 'regulator': PrincipleWeight(
 fairness=0.30, accountability=0.25, transparency=0.20,
 privacy=0.15, safety=0.05, accuracy=0.05
 ),
 'company_leadership': PrincipleWeight(
 accuracy=0.25, fairness=0.20, accountability=0.20,
 transparency=0.15, safety=0.10, privacy=0.10
 ),
 }
 
 priority = calculate_principle_priority(
 context, stakeholder_weights, stakeholder_priorities
 )
 
 return priority

## Ethics 원칙의 문서화와 커뮤니케이션

### 윤리적 의사결정 기록

## 정답 없는 난제들: 저자의 견해

### 머신 언러닝(Machine Unlearning): ``잊힐 권리''는 실현 가능한가?

### 생성형 AI의 저작권: 창조와 도용 사이의 회색지대

### 뉴로심볼릭 AI: 거버넌스의 새로운 지평